# Basirah (بصيرة): training the CNN and the results

The project is two files, side by side in one folder:

- **`index.html`**: the app (the web version): camera, calibration, the 3x3 board, Arabic speech, and the 3x3 test. It carries my trained CNN inside it.
- **`train_and_test.ipynb`** (this notebook): how the CNN was trained and how well it works.

Parts:

- **Part 1** prepares MPIIGaze and trains / evaluates the CNN with leave-one-person-out, next to the MPIIGaze paper's network and the CNN without head pose.
- **Part 2** the 3x3 test: recorded in the app (`index.html`, key T after a calibration), its saved results are brought here.
- **Part 3** shows every result.
- **Part 4** puts a newly trained CNN into `index.html` (as ONNX), so the app always runs the latest model.

Open it with the project's Python (`.venv`) as the kernel and run the first code cell first. The run cells are switched off (`#`) so that *Run All* never starts a training by accident.

Everything these parts write (MPIIGaze, results, figures, the trained model) goes to `~/Library/Application Support/Basirah`, not next to this notebook.

# Part 1. Training the CNN


```text
Prepare MPIIGaze, then train and evaluate the CNN with leave-one-person-out (LOPO).

    train_main(["--prepare"])          # once: load, clean and save MPIIGaze
    train_main([])                     # LOPO: 15 trainings, each tested on an unseen person
    train_main(["--no-head"])          # ablation: without the head pose
    train_main(["--arch", "lenet"])    # the MPIIGaze paper's network (Zhang et al. 2015, 6.3 deg)
    train_main(["--final"])            # train on all 15 people -> assets/cnn.pt

Data: the standard Evaluation Subset, 15 people x 3000 normalised eye images (1500 left + 1500
right), in data/mpiigaze/MPIIGaze next to the app's data (~/Library/Application Support/Basirah).
Right eyes are mirrored (and their yaw negated) so the network sees one kind of eye; invalid
samples are flagged in a mask, not deleted.
Training: nothing is tuned on the test person. The schedule is fixed (AdamW, one-cycle learning
rate, L1 loss, 15 epochs) and the test error per epoch is only logged for the learning curves.
Results (in ~/Library/Application Support/Basirah/data/results): <run>_lopo.json, <run>_per_person.png, <run>_curves.png, mpiigaze_stats.json
```

## The parts of the app the training uses, and the folders

In [ ]:
# libraries, folders, and the parts of the app the training uses: the CNN and the angle conventions
from __future__ import annotations

import argparse
import base64
import json
import math
import re
import shutil
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F

USER_ROOT = Path.home() / "Library" / "Application Support" / "Basirah"
DATA = USER_ROOT / "data"  # MPIIGaze, results
ASSETS = USER_ROOT / "assets"  # the trained CNN (cnn.pt)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "train_and_test.ipynb").exists())  # this folder
INDEX_HTML = ROOT / "index.html"  # the app

In [ ]:
# gaze_vector_to_pitchyaw: 3D gaze direction -> pitch and yaw angles (pitch = asin(-y), yaw = atan2(-x, -z))
def gaze_vector_to_pitchyaw(v: np.ndarray) -> np.ndarray:
    v = np.atleast_2d(np.asarray(v, dtype=np.float64))
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    return np.stack([np.arcsin(np.clip(-v[:, 1], -1.0, 1.0)), np.arctan2(-v[:, 0], -v[:, 2])], axis=1)

In [ ]:
# head_pose_to_pitchyaw: head rotation -> head pitch and yaw
def head_pose_to_pitchyaw(rvec: np.ndarray) -> np.ndarray:
    """(N, 3) Rodrigues head rotations -> (N, 2) [pitch, yaw]."""
    out = []
    for r in np.atleast_2d(np.asarray(rvec, dtype=np.float64)):
        z = cv2.Rodrigues(r.reshape(3, 1))[0][:, 2]
        out.append((np.arcsin(np.clip(z[1], -1.0, 1.0)), np.arctan2(z[0], z[2])))
    return np.array(out)

In [ ]:
# pixel normalisation of the eye images (the same in training and in the app)
# pick_device: use the Apple GPU (MPS) when there is one
# to_tensor: eye images -> normalised tensors for the CNN
PIX_MEAN, PIX_STD = 0.5, 0.29  # histogram-equalised images are about uniform on [0, 1]


def pick_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def to_tensor(images_uint8: np.ndarray, device="cpu") -> torch.Tensor:
    """(N, 36, 60) uint8 -> (N, 1, 36, 60) normalised float (same in training and in the app)."""
    x = torch.as_tensor(np.ascontiguousarray(images_uint8), dtype=torch.float32, device=device)
    return ((x / 255.0 - PIX_MEAN) / PIX_STD).unsqueeze(1)

In [ ]:
# _block: two 3x3 convolutions, each with BatchNorm and ReLU
def _block(cin: int, cout: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )

In [ ]:
# GazeCNN: the CNN (4 blocks, FC 256, + head pose) -> gaze pitch and yaw
class GazeCNN(nn.Module):
    def __init__(self, use_head: bool = True, dropout: float = 0.3):
        super().__init__()
        self.use_head = use_head
        self.features = nn.Sequential(
            _block(1, 32), nn.MaxPool2d(2),     # 36x60 -> 18x30
            _block(32, 64), nn.MaxPool2d(2),    # -> 9x15
            _block(64, 128), nn.MaxPool2d(2),   # -> 4x7
            _block(128, 128),
            nn.Conv2d(128, 32, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.fc = nn.Sequential(nn.Linear(32 * 4 * 7, 256), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.out = nn.Linear(256 + (2 if use_head else 0), 2)

    def forward(self, x: torch.Tensor, head: torch.Tensor | None = None) -> torch.Tensor:
        x = self.fc(self.features(x).flatten(1))
        if self.use_head:
            x = torch.cat([x, head], dim=1)
        return self.out(x)

In [2]:
# where MPIIGaze is, where the prepared data and results go, and the cleaning limits
DATASET = DATA / "mpiigaze" / "MPIIGaze"
PREPARED = DATA / "mpiigaze" / "prepared" / "eval_subset.npz"
RESULTS = DATA / "results"  # ~/Library/Application Support/Basirah/data/results
MAX_GAZE_DEG = 50.0  # a laptop screen spans about +-25 degrees
MIN_IMAGE_STD = 3.0  # an eye image with almost no contrast is broken

In [ ]:
# pack_web: the trained CNN (cnn.pt) -> ONNX -> inside index.html, so the app runs this model
def export_onnx(model: nn.Module) -> bytes:
    """The CNN as ONNX bytes, checked against PyTorch on random eye images."""
    import io
    import onnxruntime as ort  # pip install onnx onnxruntime onnxscript
    buf = io.BytesIO()
    x, h = torch.zeros(2, 1, 36, 60), torch.zeros(2, 2)
    model = model.cpu().eval()
    torch.onnx.export(model, (x, h), buf, dynamo=False, input_names=["eyes", "head"], output_names=["gaze"],
                      opset_version=17, dynamic_axes={"eyes": {0: "n"}, "head": {0: "n"}, "gaze": {0: "n"}})
    rng = np.random.default_rng(0)
    xs = ((rng.integers(0, 256, (64, 1, 36, 60)) / 255.0 - PIX_MEAN) / PIX_STD).astype(np.float32)
    hs = rng.normal(0, 0.2, (64, 2)).astype(np.float32)
    with torch.no_grad():
        ref = model(torch.from_numpy(xs), torch.from_numpy(hs)).numpy()
    got = ort.InferenceSession(buf.getvalue()).run(["gaze"], {"eyes": xs, "head": hs})[0]
    diff = float(np.degrees(np.abs(got - ref)).max())
    if diff > 0.01:
        raise ValueError(f"the ONNX model differs from PyTorch by {diff:.4f} degrees")
    print(f"  ONNX: {len(buf.getvalue()) / 1e6:.1f} MB, max difference to PyTorch {diff:.6f} degrees")
    return buf.getvalue()


def pack_web(checkpoint: Path = ASSETS / "cnn.pt", page: Path = INDEX_HTML) -> None:
    """Replace the CNN inside index.html (its embedded files, at the end) with this checkpoint."""
    ckpt = torch.load(checkpoint, map_location="cpu", weights_only=False)
    model = GazeCNN(ckpt["config"].get("use_head", True))
    model.load_state_dict(ckpt["state_dict"])
    onnx_bytes = export_onnx(model)
    html = page.read_text(encoding="utf-8")
    m = re.search(r'(<script id="basirah-files" type="application/json">)(.*?)(</script>)', html, re.S)
    files = json.loads(m.group(2))
    files["cnn.onnx"] = base64.b64encode(onnx_bytes).decode()
    page.write_text(html[:m.start(2)] + json.dumps(files, separators=(",", ":")) + html[m.end(2):], encoding="utf-8")
    print(f"  {page.name}: the CNN inside it is now {checkpoint.name} ({page.stat().st_size / 1e6:.1f} MB)")

## 1. Preparing MPIIGaze

In [3]:
# load_day: one day of MPIIGaze (eye images, gaze, head pose)
def load_day(path: Path) -> dict:
    mat = scipy.io.loadmat(str(path), struct_as_record=False, squeeze_me=True)
    names = np.atleast_1d(np.asarray(mat["filenames"])).astype(str)
    n = len(names)
    out = {"filenames": names}
    for side in ("left", "right"):
        s = getattr(mat["data"], side)
        out[side] = {"image": np.asarray(s.image).reshape(n, 36, 60), "gaze": np.asarray(s.gaze).reshape(n, 3),
                     "pose": np.asarray(s.pose).reshape(n, 3)}
    return out

In [4]:
# load_person: one person's 3000 eye images of the Evaluation Subset
def load_person(root: Path, person: int) -> dict:
    pid = f"p{person:02d}"
    rows = [line.split() for line in (root / "Evaluation Subset" / "sample list for eye image" / f"{pid}.txt")
            .read_text().splitlines() if line.strip()]  # "day08/0069.jpg left"
    days = {d: load_day(root / "Data" / "Normalized" / pid / f"{d}.mat") for d in sorted({r[0].split("/")[0] for r in rows})}
    index = {d: {name: i for i, name in enumerate(v["filenames"])} for d, v in days.items()}
    out = {k: [] for k in ("images", "gaze", "head", "side", "day", "filename")}
    for rel, side in rows:
        day, fname = rel.split("/")
        s, i = days[day][side], index[day][fname]
        img = s["image"][i]
        g, h = gaze_vector_to_pitchyaw(s["gaze"][i])[0], head_pose_to_pitchyaw(s["pose"][i])[0]
        if side == "right":  # mirror, so every sample looks like a left eye
            img, g, h = img[:, ::-1], g * [1.0, -1.0], h * [1.0, -1.0]
        for k, v in zip(out, (np.ascontiguousarray(img), g, h, int(side == "right"), int(day[3:]), fname)):
            out[k].append(v)
    out["person"] = [person] * len(rows)
    return out

In [5]:
# prepare: load, clean and save the Evaluation Subset (run once)
def prepare(dataset: Path) -> int:
    """Load, clean, check the mirroring and save the Evaluation Subset."""
    if not (dataset / "Data" / "Normalized").exists():
        print(f"MPIIGaze not found at {dataset}")
        return 1
    parts = []
    for p in range(15):
        parts.append(load_person(dataset, p))
        print(f"  p{p:02d}: {len(parts[-1]['images'])} samples", flush=True)
    s = {k: np.concatenate([np.asarray(part[k]) for part in parts]) for k in parts[0]}
    s["images"], s["gaze"], s["head"] = s["images"].astype(np.uint8), s["gaze"].astype(np.float32), s["head"].astype(np.float32)
    n = len(s["images"])

    # clean: flag, don't delete
    finite = np.isfinite(s["gaze"]).all(axis=1) & np.isfinite(s["head"]).all(axis=1)
    key = [f"{a}/{b}/{c}/{d}" for a, b, c, d in zip(s["person"], s["day"], s["filename"], s["side"])]
    unique = np.zeros(n, bool)
    unique[np.unique(key, return_index=True)[1]] = True
    not_blank = s["images"].reshape(n, -1).std(axis=1) >= MIN_IMAGE_STD
    plausible = (np.abs(np.degrees(s["gaze"])) <= MAX_GAZE_DEG).all(axis=1)
    valid = finite & unique & not_blank & plausible
    print(f"{n} samples: {(~finite).sum()} non-finite, {(~unique).sum()} duplicates, {(~not_blank).sum()} blank, "
          f"{(~plausible).sum()} implausible -> {valid.sum()} valid")

    # check: the mean left eye must look like the mean MIRRORED right eye
    left, right = s["images"][(s["side"] == 0) & valid].mean(axis=0), s["images"][(s["side"] == 1) & valid].mean(axis=0)
    mirrored, unmirrored = np.corrcoef(left.ravel(), right.ravel())[0, 1], np.corrcoef(left.ravel(), right[:, ::-1].ravel())[0, 1]
    print(f"mean left vs mirrored right eye r={mirrored:.3f} (un-mirrored r={unmirrored:.3f}) -> "
          + ("mirroring OK" if mirrored > unmirrored else "MIRRORING WRONG"))

    PREPARED.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(PREPARED, **s)
    np.save(PREPARED.with_name("eval_subset_valid.npy"), valid)
    RESULTS.mkdir(parents=True, exist_ok=True)
    (RESULTS / "mpiigaze_stats.json").write_text(json.dumps({
        "samples": int(n), "valid": int(valid.sum()), "non_finite": int((~finite).sum()), "duplicates": int((~unique).sum()),
        "blank": int((~not_blank).sum()), "implausible_gaze": int((~plausible).sum()),
        "mean_image_corr_mirrored": float(mirrored), "mean_image_corr_unmirrored": float(unmirrored)}, indent=2))
    print(f"saved {PREPARED}")
    return 0

## 2. Training

In [6]:
# LeNetGaze: the network of the MPIIGaze paper, for comparison
class LeNetGaze(nn.Module):
    """The MPIIGaze paper baseline: 2 conv layers + FC 500, head pose before the last layer."""

    def __init__(self, use_head: bool = True):
        super().__init__()
        self.use_head = use_head
        self.conv1, self.conv2 = nn.Conv2d(1, 20, 5), nn.Conv2d(20, 50, 5)
        self.fc1, self.fc2 = nn.Linear(50 * 6 * 12, 500), nn.Linear(500 + (2 if use_head else 0), 2)

    def forward(self, x, head=None):
        x = F.relu(self.fc1(F.max_pool2d(self.conv2(F.max_pool2d(self.conv1(x), 2)), 2).flatten(1)))
        return self.fc2(torch.cat([x, head], dim=1) if self.use_head else x)

In [7]:
# TrainConfig: the training settings (15 epochs, batch 256, AdamW, lr 0.001)
@dataclass
class TrainConfig:
    arch: str = "cnn"
    use_head: bool = True
    augment: bool = True
    epochs: int = 15
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 0

In [8]:
# augment: small shifts, zoom, blur and noise, so MPIIGaze looks like a live webcam
def augment(x: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
    """Make MPIIGaze images look like live webcam crops, without changing where the eye looks:
    small shift/zoom (the live eye is not perfectly centred), blur on 30%, sensor noise."""
    n, dev = x.shape[0], x.device

    def rand(*shape):
        return torch.rand(*shape, generator=gen, device="cpu").to(dev)

    scale = 1.0 + (rand(n) - 0.5) * 0.14
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0] = theta[:, 1, 1] = scale
    theta[:, 0, 2], theta[:, 1, 2] = (rand(n) - 0.5) * 0.12, (rand(n) - 0.5) * 0.16
    x = F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False), mode="bilinear",
                      padding_mode="border", align_corners=False)
    blur = rand(n) < 0.3
    if blur.any():
        sigma = 0.4 + float(rand(1)) * 0.6
        k = torch.exp(-(torch.arange(-1, 2, device=dev, dtype=torch.float32) ** 2) / (2 * sigma ** 2))
        k = k / k.sum()
        x = torch.where(blur.view(n, 1, 1, 1),
                        F.conv2d(F.pad(x, (1, 1, 1, 1), mode="replicate"), (k[:, None] * k[None, :]).view(1, 1, 3, 3)), x)
    noise_std = rand(n).view(n, 1, 1, 1) * (0.05 / PIX_STD)  # up to ~13/255 grey levels
    return x + torch.randn(x.shape, generator=gen, device="cpu").to(dev) * noise_std

In [9]:
# angular_error_deg: the angle between predicted and true gaze (degrees)
# predict: the model's gaze for many images, in batches
def angular_error_deg(pred: torch.Tensor, true: torch.Tensor) -> torch.Tensor:
    def vec(py):
        p, y = py[:, 0], py[:, 1]
        return torch.stack([-torch.cos(p) * torch.sin(y), -torch.sin(p), -torch.cos(p) * torch.cos(y)], 1)
    return torch.rad2deg(torch.acos((vec(pred) * vec(true)).sum(1).clamp(-1.0, 1.0)))


@torch.no_grad()
def predict(model: nn.Module, x: torch.Tensor, head: torch.Tensor, batch: int = 2048) -> torch.Tensor:
    model.eval()
    return torch.cat([model(x[i:i + batch], head[i:i + batch]) for i in range(0, len(x), batch)])

In [10]:
# train: trains one model (L1 loss, one-cycle learning rate)
def train(cfg: TrainConfig, x, h, y, test=None, on_epoch=None):
    """-> (model, train loss per epoch, test error per epoch)."""
    torch.manual_seed(cfg.seed)
    gen = torch.Generator().manual_seed(cfg.seed)
    dev = x.device
    model = (GazeCNN(cfg.use_head) if cfg.arch == "cnn" else LeNetGaze(cfg.use_head)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=cfg.epochs * math.ceil(len(x) / cfg.batch_size),
                                                pct_start=0.1, anneal_strategy="cos")
    losses, test_errs, t0 = [], [], time.perf_counter()
    for epoch in range(cfg.epochs):
        model.train()
        perm = torch.randperm(len(x), generator=gen).to(dev)
        total = torch.zeros((), device=dev)  # summed on the device: no GPU->CPU sync per step
        for i in range(0, len(x), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            loss = F.l1_loss(model(augment(x[idx], gen) if cfg.augment else x[idx], h[idx]), y[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            total += loss.detach() * len(idx)
        losses.append(float(total) / len(x))
        if test is not None:
            test_errs.append(float(angular_error_deg(predict(model, test[0], test[1]), test[2]).mean()))
        if on_epoch is not None:
            on_epoch(epoch, losses, time.perf_counter() - t0)
    return model, losses, test_errs

In [11]:
# plot: the per-person errors and the learning curves
def plot(out_dir: Path, name: str, folds: dict, epochs: int) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    errs = np.array([f["error_deg"] for f in folds.values()])
    fig, ax = plt.subplots(figsize=(9, 3.6), constrained_layout=True)
    ax.bar(list(folds), errs, color="#2a6fbb", width=0.6)
    ax.axhline(errs.mean(), color="#444444", linewidth=1, linestyle="--")
    ax.set(ylabel="mean angular error (deg)", title=f"{name}: error on each unseen person (mean {errs.mean():.2f}°)")
    fig.savefig(out_dir / f"{name}_per_person.png", dpi=150)
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), constrained_layout=True)
    ep = np.arange(1, epochs + 1)
    for ax, k, title in ((axes[0], "curve_train_l1_deg", "training loss (L1 per angle, deg)"),
                         (axes[1], "curve_test_err_deg", "error on the unseen person (deg)")):
        ax.plot(ep, np.mean([f[k] for f in folds.values()], axis=0), color="#2a6fbb", linewidth=2)
        ax.set(title=title, xlabel="epoch")
    fig.savefig(out_dir / f"{name}_curves.png", dpi=150)
    plt.close(fig)

In [12]:
# train_main: prepare, leave-one-person-out, or the final model
def train_main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Prepare MPIIGaze, train / evaluate the CNN (leave-one-person-out)")
    parser.add_argument("--arch", default="cnn", choices=["cnn", "lenet"])
    parser.add_argument("--no-head", action="store_true")
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--folds", default="all", help="'all' or test people, e.g. 0,5,9")
    parser.add_argument("--final", action="store_true", help="train on all 15 people and save the model")
    parser.add_argument("--prepare", action="store_true", help="prepare MPIIGaze (once, before training)")
    parser.add_argument("--dataset", type=Path, default=DATASET, help="where MPIIGaze is")
    args = parser.parse_args(argv)
    if args.prepare:
        return prepare(args.dataset)
    cfg = TrainConfig(arch=args.arch, use_head=not args.no_head, epochs=args.epochs)
    name = cfg.arch + ("" if cfg.use_head else "_nohead")
    if not PREPARED.exists():
        print(f"{PREPARED} not found - run: train_main(['--prepare'])")
        return 1
    d, dev = np.load(PREPARED), pick_device()
    valid_path = PREPARED.with_name("eval_subset_valid.npy")
    valid = np.load(valid_path) if valid_path.exists() else np.ones(len(d["images"]), bool)
    X, person = to_tensor(d["images"], dev), d["person"].astype(int)
    H = torch.as_tensor(d["head"], dtype=torch.float32, device=dev)
    Y = torch.as_tensor(d["gaze"], dtype=torch.float32, device=dev)
    print(f"{valid.sum()} eye images, {len(np.unique(person))} people, device {dev}, run {name}")

    if args.final:
        idx = torch.as_tensor(np.flatnonzero(valid), device=dev)
        model, losses, _ = train(cfg, X[idx], H[idx], Y[idx], on_epoch=lambda e, l, s: print(
            f"  epoch {e + 1:2d}/{cfg.epochs}  train L1 {np.degrees(l[-1]):.2f} deg  ({s / 60:.1f} min)", flush=True))
        path = ASSETS / f"{name}.pt"
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "config": asdict(cfg),
                    "pix_mean": PIX_MEAN, "pix_std": PIX_STD,
                    "info": {"trained_on": "MPIIGaze Evaluation Subset, all 15 people", "samples": int(len(idx)),
                             "date": time.strftime("%Y-%m-%d %H:%M")}}, path)
        print(f"saved {path}")
        if name == "cnn":
            pack_web(path)  # the app (index.html) runs the new model
        return 0

    people = sorted(np.unique(person)) if args.folds == "all" else [int(p) for p in args.folds.split(",")]
    out_dir = RESULTS
    out_dir.mkdir(parents=True, exist_ok=True)
    results = {"run": name, "config": asdict(cfg), "device": str(dev), "folds": {}}
    for p in people:
        tr = torch.as_tensor(np.flatnonzero(valid & (person != p)), device=dev)
        te = torch.as_tensor(np.flatnonzero(valid & (person == p)), device=dev)
        model, losses, test_errs = train(cfg, X[tr], H[tr], Y[tr], test=(X[te], H[te], Y[te]))
        err = angular_error_deg(predict(model, X[te], H[te]), Y[te]).cpu().numpy()
        base = float(angular_error_deg(Y[tr].mean(0, keepdim=True).expand(len(te), 2), Y[te]).mean())  # always the mean gaze
        results["folds"][f"p{p:02d}"] = {"test_n": int(len(te)), "error_deg": float(err.mean()), "mean_predictor_deg": base,
                                         "curve_train_l1_deg": [float(np.degrees(v)) for v in losses],
                                         "curve_test_err_deg": test_errs}
        print(f"  p{p:02d}: {err.mean():.2f}° (always the mean gaze: {base:.2f}°)", flush=True)
        (out_dir / f"{name}_lopo.json").write_text(json.dumps(results, indent=2))  # saved after every fold
    errs = np.array([f["error_deg"] for f in results["folds"].values()])
    results.update(mean_error_deg=float(errs.mean()), std_error_deg=float(errs.std()),
                   mean_predictor_deg=float(np.mean([f["mean_predictor_deg"] for f in results["folds"].values()])))
    (out_dir / f"{name}_lopo.json").write_text(json.dumps(results, indent=2))
    print(f"\n{name}: {errs.mean():.2f}° ± {errs.std():.2f}° over {len(errs)} unseen people "
          f"(always the mean gaze: {results['mean_predictor_deg']:.2f}°)")
    plot(out_dir, name, results["folds"], cfg.epochs)
    return 0

## Run

Switched off so that *Run All* does not start them: to run one, remove the `#` at the start of its last line.

In [13]:
# once: load, clean and save MPIIGaze (in ~/Library/Application Support/Basirah/data/mpiigaze/MPIIGaze)
# train_main(["--prepare"])

In [14]:
# leave-one-person-out: 15 trainings, each tested on an unseen person (about 70 min); also ["--no-head"], ["--arch", "lenet"], ["--folds", "0,5,9"]
# train_main([])

In [15]:
# the final model on all 15 people (about 5 min) - this REPLACES the app's CNN and puts it into index.html
# train_main(["--final"])

# Part 2. The 3x3 test

The project's main metric, recorded in the app itself (`index.html`), so the number measures exactly what the user gets.

```text
In the app: a full calibration, then T (or the "3x3 test" button on the result screen).
Nine words in a 3x3 grid; one cell at a time is lit for 3 s and you look at its word, 36 times.
No gaze cursor is shown on purpose, so you cannot steer it: the recording measures the model.
The app then replays every frame through its own gaze path, simulates the dwell selection in
every trial, and shows the hit rate with a 95% confidence interval, the errors, and what the
filter and the dwell time change. "Save the result (JSON)" downloads it.
```

The scoring is the same as the Python version's `grid_test.py` (checked: the same numbers on the same recording). The cell below brings the saved tests from Downloads into the results.

In [ ]:
# import_web_tests: the 3x3 tests saved by the app (Downloads/basirah_test_*.json) -> results
def import_web_tests(folder: Path = Path.home() / "Downloads") -> None:
    RESULTS.mkdir(parents=True, exist_ok=True)
    found = sorted(folder.glob("basirah_test_*.json"))
    for f in found:
        r = json.loads(f.read_text(encoding="utf-8"))
        shutil.copy2(f, RESULTS / f"test_{r['test']}.json")
        s = r["score"]
        print(f"  {r['test']}: hit {s['hit_rate']:.0f}% ({s['hits']}/{s['trials']})  [{r.get('method', '')}]")
    if not found:
        print(f"no basirah_test_*.json in {folder}: in the app, after a test, press 'Save the result (JSON)'")


import_web_tests()

# Part 3. Results

Everything the training and the 3x3 tests saved in `~/Library/Application Support/Basirah/data/results` (Python version and web version): the CNN on unseen people (with the two comparisons), then every 3x3 test, then the figures. This part only reads files, so it is safe to run.


In [ ]:
# lopo_summary: the CNN's error on unseen people, next to LeNet and the CNN without head pose
RUNS = {"cnn": "my CNN (with head pose)", "cnn_nohead": "my CNN, no head pose", "lenet": "LeNet (MPIIGaze paper)"}


def lopo_summary() -> None:
    print("CNN - error on an unseen person (leave-one-person-out, MPIIGaze, 15 people)")
    found = False
    for run, label in RUNS.items():
        path = RESULTS / f"{run}_lopo.json"
        if not path.exists():
            continue
        r = json.loads(path.read_text())
        if "mean_error_deg" not in r:
            print(f"  {label:28s} unfinished ({len(r['folds'])}/15 people)")
            continue
        found = True
        errs = {k: f["error_deg"] for k, f in r["folds"].items()}
        worst = max(errs, key=errs.get)
        print(f"  {label:28s} {r['mean_error_deg']:.2f}° ± {r['std_error_deg']:.2f}°   best {min(errs.values()):.2f}°  "
              f"worst {errs[worst]:.2f}° ({worst})   always-the-mean {r['mean_predictor_deg']:.2f}°")
    if not found:
        print("  no results yet: train_main([]) (about 70 min), and for the comparisons "
              "train_main(['--arch', 'lenet']) and train_main(['--no-head'])")
    stats = RESULTS / "mpiigaze_stats.json"
    if stats.exists():
        s = json.loads(stats.read_text())
        print(f"  data: {s['valid']} of {s['samples']} eye images valid")


In [ ]:
# grid_summary: every recorded 3x3 test
def grid_summary() -> None:
    tests = sorted(RESULTS.glob("test_*.json"))
    print(f"3x3 tests ({len(tests)})")
    for path in tests:
        r = json.loads(path.read_text(encoding="utf-8"))
        s, e = r["score"], r["errors"]["filtered"]
        lo, hi = r["hit_rate_ci95"]
        t = s.get("mean_select_time_s")
        when = f"select time {t:.2f} s" if t is not None else "no selection"
        print(f"  {r['test']}: hit {s['hit_rate']:.0f}% ({s['hits']}/{s['trials']}, CI {lo:.0f}-{hi:.0f}%)  "
              f"wrong {s['wrong_rate']:.0f}%  {when}  "
              f"median error {e['median_err_pt']:.0f} pt (bias {e['bias_pt']:.0f}, jitter {e['jitter_pt']:.0f})")


In [ ]:
# the numbers, then the figures
from IPython.display import Image, display

lopo_summary()
print()
grid_summary()
for png in sorted(RESULTS.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))


# Part 4. The app runs the latest CNN

`train_main(["--final"])` puts the new model into `index.html` by itself. To do it by hand (e.g. after copying `cnn.pt` from another Mac), remove the `#` below and run the cell. It converts the CNN to ONNX, checks it against PyTorch (the same angles to 0.01 degrees), and replaces the model inside `index.html`.

In [ ]:
# put assets/cnn.pt into index.html (needs: pip install onnx onnxruntime onnxscript)
# pack_web()